<a id="topsis"></a>

$$
\text{TOPSIS 法 (Technique for Order Preference by Similarity to Ideal Solution, 优劣解距离法)} \\
\text{评价类模型: 有 n 个评价对象, m 个评价指标, 给每个对象打分并排序} \\ \quad \\
\text{核心思想: 好的方案应该离 "最优解" 尽可能近, 同时离 "最劣解" 尽可能远} \\ \quad \\
\text{步骤:} \\
\text{1. 正向化 (Forward Processing): 把所有指标都变成 "越大越好"} \\
\text{2. 标准化 (Normalization): 消除量纲 (单位) 的影响} \\
\text{3. 找到最优解 } Z^+ \text{ 和最劣解 } Z^- \\
\text{4. 计算每个对象到 } Z^+ \text{ 和 } Z^- \text{ 的距离 } D^+, D^- \\
\text{5. 得分 } C_i = \frac{D_i^-}{D_i^+ + D_i^-} \text{, 按得分排序} \\ \quad \\
\text{和层次分析法 (AHP) 的区别: AHP 的判断矩阵依赖主观打分, TOPSIS 直接利用原始数据}
$$

<a id="forward-processing"></a>

$$
\text{1. 正向化: 四种指标类型} \\ \quad \\
\text{极大型 (Benefit): 越大越好, 例如成绩, 含氧量 } \implies \text{不变} \\ \quad \\
\text{极小型 (Cost): 越小越好, 例如费用, 细菌数} \\
\text{需要一个单调递减的变换, 把最小值变成最大值: } \tilde{x}_i = \max_k x_k - x_i \\
\text{(若数据全为正, 也可以用 } \tilde{x}_i = \frac{1}{x_i} \text{, 但会改变数据间的相对距离)} \\ \quad \\
\text{中间型 (Intermediate): 越接近某个最佳值 } x_{best} \text{ 越好, 例如 PH 值越接近 7 越好} \\
|x_i - x_{best}| \text{ 是 "离最佳值的距离", 这是一个极小型指标; 先除以最大距离 } M \text{ 变到 [0, 1], 再用 1 减去它} \\
M = \max_k |x_k - x_{best}|, \quad \tilde{x}_i = 1 - \frac{|x_i - x_{best}|}{M} \\
\text{(等于 } x_{best} \text{ 时得 1, 离得最远的得 0)} \\ \quad \\
\text{区间型 (Interval): 落在区间 [a, b] 内最好, 例如体温, 营养物含量} \\
\text{"离区间的距离" } = \begin{cases} a - x_i, & x_i < a \\ 0, & a \leq x_i \leq b \\ x_i - b, & x_i > b \end{cases} \\
\text{同样除以最大距离 } M = \max\{a - \min_k x_k, \ \max_k x_k - b\} \text{, 再用 1 减去:} \\
\tilde{x}_i = \begin{cases} 1 - \frac{a - x_i}{M}, & x_i < a \\ 1, & a \leq x_i \leq b \\ 1 - \frac{x_i - b}{M}, & x_i > b \end{cases}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def to_benefit_from_cost(x):
    return x.max() - x

def to_benefit_from_intermediate(x, best):
    M = np.max(np.abs(x - best))
    return 1 - np.abs(x - best) / M

def to_benefit_from_interval(x, a, b):
    M = max(a - x.min(), x.max() - b)
    distance = np.where(x < a, a - x, np.where(x > b, x - b, 0.0))
    return 1 - distance / M

# 例子: 5 条河流的水质评价
# 含氧量 (极大型), PH 值 (中间型, 最佳 7), 细菌总数 (极小型), 植物性营养物量 (区间型, [10, 20])
rivers = ['A', 'B', 'C', 'D', 'E']
raw = np.array([[4.69, 6.59, 51, 11.94],
                [2.03, 7.86, 19, 6.46],
                [9.11, 6.31, 46, 8.91],
                [8.61, 7.05, 46, 26.43],
                [7.13, 6.50, 50, 23.57]])

X_forward = np.column_stack([
    raw[:, 0],
    to_benefit_from_intermediate(raw[:, 1], best=7),
    to_benefit_from_cost(raw[:, 2]),
    to_benefit_from_interval(raw[:, 3], a=10, b=20),
])
print(f'正向化后的矩阵 = \n{np.round(X_forward, 4)}')

# 检查: 正向化后每一列都是 "越大越好", 原来最好的对象在新的列里最大
print(f'PH 最接近 7 的是 {rivers[np.argmin(np.abs(raw[:, 1] - 7))]}, 正向化后最大的是 {rivers[np.argmax(X_forward[:, 1])]}')
print(f'细菌最少的是 {rivers[np.argmin(raw[:, 2])]}, 正向化后最大的是 {rivers[np.argmax(X_forward[:, 2])]}')

<a id="vector-normalization"></a>

$$
\text{2. 标准化: 向量归一化 (Vector Normalization)} \\
z_{ij} = \frac{\tilde{x}_{ij}}{\sqrt{\sum_{i=1}^{n} \tilde{x}_{ij}^2}} \\ \quad \\
\text{为什么: 不同指标单位不同 (mg/L, 个/mL), 数值大小差别很大, 直接算距离会被数值大的指标主导} \\
\text{除以该列的模长后, 每一列都变成单位向量: } \sum_i z_{ij}^2 = \frac{\sum_i \tilde{x}_{ij}^2}{\sum_i \tilde{x}_{ij}^2} = 1 \\
\text{性质: 某一列整体乘以常数 c (换单位), } z_{ij} = \frac{c \tilde{x}_{ij}}{\sqrt{\sum_i c^2 \tilde{x}_{ij}^2}} = \frac{\tilde{x}_{ij}}{\sqrt{\sum_i \tilde{x}_{ij}^2}} \text{ 不变} \\
\text{(也可以用 min-max 归一化, 只要能消除量纲即可)}
$$

数学基础: [归一化与标准化](../math_basics/statistics.ipynb#normalization)

In [ ]:
Z = X_forward / np.sqrt(np.sum(X_forward**2, axis=0))
print(f'标准化后的矩阵 Z = \n{np.round(Z, 4)}')
print(f'每一列的模长 = {np.round(np.linalg.norm(Z, axis=0), 6)}')

# 和 sklearn 的按列 L2 归一化对比 (sklearn 默认按行, 所以转置)
from sklearn.preprocessing import normalize
print(f'== sklearn normalize(axis=0) ? {np.allclose(Z, normalize(X_forward, norm="l2", axis=0))}')

# 换单位不影响结果: 含氧量 mg/L -> ug/L
X_unit = X_forward.copy()
X_unit[:, 0] *= 1000
Z_unit = X_unit / np.sqrt(np.sum(X_unit**2, axis=0))
print(f'含氧量换单位后 Z 不变? {np.allclose(Z, Z_unit)}')

<a id="topsis-score"></a>

$$
\text{3. 最优解和最劣解: 每一列分别取最大值和最小值 (已经正向化, 最大就是最好)} \\
Z^+ = (\max_i z_{i1}, \max_i z_{i2}, \cdots, \max_i z_{im}), \quad Z^- = (\min_i z_{i1}, \cdots, \min_i z_{im}) \\
\text{注意: } Z^+ \text{ 往往不是某个真实存在的对象, 而是把每个指标的最好值拼起来的 "虚拟最优方案"} \\ \quad \\
\text{4. 欧氏距离 (带权重 } w_j, \sum w_j = 1 \text{):} \\
D_i^+ = \sqrt{\sum_{j=1}^m w_j (Z_j^+ - z_{ij})^2}, \quad D_i^- = \sqrt{\sum_{j=1}^m w_j (Z_j^- - z_{ij})^2} \\ \quad \\
\text{5. 得分: } C_i = \frac{D_i^-}{D_i^+ + D_i^-} \\
\text{为什么在 [0, 1] 之间: } D_i^+ \geq 0, D_i^- \geq 0 \implies 0 \leq D_i^- \leq D_i^+ + D_i^- \implies 0 \leq C_i \leq 1 \\
C_i = 1 \iff D_i^+ = 0 \text{ (就是最优解)}, \quad C_i = 0 \iff D_i^- = 0 \text{ (就是最劣解)} \\
C_i \text{ 关于 } D_i^- \text{ 递增, 关于 } D_i^+ \text{ 递减 (对 } D^+ \text{ 求偏导: } \frac{\partial C}{\partial D^+} = -\frac{D^-}{(D^+ + D^-)^2} \leq 0 \text{)} \\ \quad \\
\text{为什么不只用 } D^+ \text{? 离最优解最近的点不一定离最劣解最远 (见下图: P2 离 } Z^+ \text{ 更近, 但 P1 离 } Z^- \text{ 更远, 最终 P1 得分更高)} \\
\text{两个距离结合, 排序更稳定}
$$

In [ ]:
def topsis(X_forward, weights=None):
    n, m = X_forward.shape
    w = np.full(m, 1 / m) if weights is None else np.asarray(weights) / np.sum(weights)
    Z = X_forward / np.sqrt(np.sum(X_forward**2, axis=0))
    Z_plus, Z_minus = Z.max(axis=0), Z.min(axis=0)
    D_plus = np.sqrt(np.sum(w * (Z_plus - Z)**2, axis=1))
    D_minus = np.sqrt(np.sum(w * (Z_minus - Z)**2, axis=1))
    return D_minus / (D_plus + D_minus), D_plus, D_minus

score, D_plus, D_minus = topsis(X_forward)
for name, s, dp, dm in zip(rivers, score, D_plus, D_minus):
    print(f'河流 {name}: D+ = {dp:.4f}, D- = {dm:.4f}, C = {s:.4f}')
print(f'排名 (从好到差): {[rivers[i] for i in np.argsort(-score)]}')

# 独立验证: 用 scipy 的加权欧氏距离 (cdist 的 w 参数) 重新计算
from scipy.spatial.distance import cdist
w_eq = np.full(4, 0.25)
Zc = X_forward / np.linalg.norm(X_forward, axis=0)
dp_check = cdist(Zc, Zc.max(axis=0, keepdims=True), metric='euclidean', w=w_eq).ravel()
dm_check = cdist(Zc, Zc.min(axis=0, keepdims=True), metric='euclidean', w=w_eq).ravel()
print(f'D+ == cdist ? {np.allclose(D_plus, dp_check)}, D- == cdist ? {np.allclose(D_minus, dm_check)}')

# 性质检查: 一个各指标都不比别人差的对象 (占优) 得分最高; 最优解本身得 1, 最劣解得 0
X_ext = np.vstack([X_forward, X_forward.max(axis=0), X_forward.min(axis=0)])
s_ext, _, _ = topsis(X_ext)
print(f'加入 "每项都最好" 和 "每项都最差" 两个虚拟对象: 得分 = {np.round(s_ext[-2:], 6)}')

In [ ]:
# 二维示意: 为什么要同时看 D+ 和 D-
Zp, Zm = np.array([1.0, 1.0]), np.array([0.0, 0.0])
P1, P2 = np.array([1.0, 0.4]), np.array([0.6, 0.6])     # P2 离 Z+ 更近, P1 离 Z- 更远
for name, p in [('P1', P1), ('P2', P2)]:
    dp, dm = np.linalg.norm(Zp - p), np.linalg.norm(Zm - p)
    print(f'{name}: D+ = {dp:.4f}, D- = {dm:.4f}, C = {dm / (dp + dm):.4f}')

plt.figure(figsize=(4, 3))
plt.plot(*Zp, 'g*', markersize=12, label='Z+ (ideal)')
plt.plot(*Zm, 'rX', markersize=10, label='Z- (negative ideal)')
for name, p, c in [('P1', P1, 'b'), ('P2', P2, 'm')]:
    plt.plot(*p, 'o', color=c, label=name)
    plt.plot([p[0], Zp[0]], [p[1], Zp[1]], '--', color=c)
    plt.plot([p[0], Zm[0]], [p[1], Zm[1]], ':', color=c)
plt.gca().set_aspect('equal')
plt.xlabel('indicator 1')
plt.ylabel('indicator 2')
plot_nicely()
plt.show()

<a id="weighted-topsis"></a>

$$
\text{带权重的 TOPSIS: 上面默认每个指标同等重要 } w_j = \frac{1}{m} \\
\text{权重的来源:} \\
\text{1. 主观赋权: 层次分析法 (AHP), 专家打分} \\
\text{2. 客观赋权: 熵权法 (根据数据本身的离散程度), 变异系数法} \\ \quad \\
\text{权重放在距离里: } D_i^+ = \sqrt{\sum_j w_j (Z_j^+ - z_{ij})^2} \\
\text{等价于先把第 j 列乘以 } \sqrt{w_j} \text{ 再算普通的欧氏距离 (所以权重越大, 这个指标在距离中的影响越大)}
$$

相关模型: [层次分析法 AHP](analytic_hierarchy_process.ipynb) · [熵权法](entropy_weight_method.ipynb)

In [ ]:
weights = np.array([0.4, 0.1, 0.3, 0.2])    # 例如来自 AHP: 含氧量最重要
score_w, _, _ = topsis(X_forward, weights)
print(f'加权得分 = {np.round(score_w, 4)}')
print(f'加权排名: {[rivers[i] for i in np.argsort(-score_w)]}, 等权排名: {[rivers[i] for i in np.argsort(-score)]}')

# 验证: 权重放在距离里 == 列乘以 sqrt(w) 后算普通欧氏距离
Zs = (X_forward / np.linalg.norm(X_forward, axis=0)) * np.sqrt(weights / weights.sum())
dp_s = np.linalg.norm(Zs.max(axis=0) - Zs, axis=1)
dm_s = np.linalg.norm(Zs.min(axis=0) - Zs, axis=1)
print(f'== 列乘以 sqrt(w) 的结果? {np.allclose(score_w, dm_s / (dp_s + dm_s))}')

x_pos = np.arange(len(rivers))
plt.figure(figsize=(4, 3))
plt.bar(x_pos - 0.2, score, width=0.4, label='equal weights')
plt.bar(x_pos + 0.2, score_w, width=0.4, label='AHP-like weights')
plt.xticks(x_pos, rivers)
plt.ylabel('TOPSIS score C')
plot_nicely()
plt.show()